# Vehicle Insurance Cross-Sell — EDA from Scratch

**Business Problem:**  
An insurance company has existing health insurance customers.  
They want to predict which customers are likely to **also buy vehicle insurance**.  
This is called **cross-selling** — selling a new product to an existing customer.

**Our job:** Build a model that outputs 1 (interested) or 0 (not interested) for each customer.

---
### Notebook Structure
1. Load & First Look
2. Data Quality Check
3. Target Variable Analysis
4. Feature-by-Feature Analysis
5. Feature Relationships with Target
6. Correlation Analysis
7. Data Preprocessing
8. Train-Test Split
9. Model Training + Evaluation
10. What to fix before production

## Step 0 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    ConfusionMatrixDisplay
)

sns.set(style='whitegrid')
pd.set_option('display.max_columns', None)  # show all columns

import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully')

---
## Step 1 — Load Data & First Look

First thing in any project: **load the data and get a feel for its shape and structure.**

In [ ]:
df = pd.read_csv('data.csv')

print(f'Shape: {df.shape}')   # (rows, columns)
print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]}')

In [ ]:
# First 5 rows — always do this before anything else
df.head()

In [ ]:
# Column names, datatypes, memory usage
df.info()

In [ ]:
# Summary statistics for all numeric columns
# This tells you the range, mean, and spread of each numeric feature
df.describe().T   # .T = transpose so it's easier to read

### Column Dictionary

| Column | Type | Meaning |
|---|---|---|
| `id` | int | Unique customer ID — not a feature, drop before modeling |
| `Gender` | str | Male / Female |
| `Age` | int | Customer age |
| `Driving_License` | 0/1 | Does the customer have a driving license? |
| `Region_Code` | float | Anonymized geographic region |
| `Previously_Insured` | 0/1 | Does customer already have vehicle insurance? |
| `Vehicle_Age` | str | Age of vehicle: `< 1 Year`, `1-2 Year`, `> 2 Years` |
| `Vehicle_Damage` | str | Has vehicle been damaged before? Yes/No |
| `Annual_Premium` | float | Yearly amount customer pays for health insurance |
| `Policy_Sales_Channel` | float | Code for how salesperson contacted customer |
| `Vintage` | int | Days customer has been with the company |
| `Response` | 0/1 | **TARGET** — 1 = interested in vehicle insurance, 0 = not interested |

---
## Step 2 — Data Quality Check

Before doing any analysis, always check:
- Are there missing values?
- Are there duplicates?
- Do the value ranges make sense?

In [ ]:
# Check for null/missing values in each column
null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / len(df)) * 100

null_report = pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})
print(null_report)
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

In [ ]:
# Check for duplicate rows
dupe_count = df.duplicated().sum()
print(f'Duplicate rows: {dupe_count}')

In [ ]:
# Check unique values in categorical columns
# This tells you whether there are unexpected values or typos
cat_cols = ['Gender', 'Vehicle_Age', 'Vehicle_Damage']
for col in cat_cols:
    print(f'{col}: {df[col].unique()}')

In [ ]:
# Check Driving_License — should only be 0 or 1
print('Driving_License values:', df['Driving_License'].value_counts().to_dict())
print('Previously_Insured values:', df['Previously_Insured'].value_counts().to_dict())
print('Response values:', df['Response'].value_counts().to_dict())

---
## Step 3 — Target Variable Analysis

Always understand your target variable first.  
The key question: **is the dataset balanced?**

If 90% of rows are class 0, a dumb model that always predicts 0 gets 90% accuracy — which is misleading.

In [ ]:
target_counts = df['Response'].value_counts()
target_pct = df['Response'].value_counts(normalize=True) * 100

print('=== Target Distribution ===')
print(pd.DataFrame({'count': target_counts, 'percentage': target_pct.round(2)}))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
axes[0].bar(['Not Interested (0)', 'Interested (1)'], target_counts, color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Response Count')
axes[0].set_ylabel('Count')
for i, v in enumerate(target_counts):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(target_counts, labels=['Not Interested', 'Interested'], autopct='%1.1f%%',
            colors=['#e74c3c', '#2ecc71'], startangle=90)
axes[1].set_title('Response Distribution')

plt.suptitle('Target Variable: Response', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Imbalance ratio
ratio = target_counts[0] / target_counts[1]
print(f'\nImbalance ratio: {ratio:.1f}:1  (for every 1 interested, {ratio:.1f} are not interested)')
print('=> This is a CLASS IMBALANCE problem. Accuracy alone is not a good metric.')

---
## Step 4 — Feature-by-Feature Analysis (Univariate)

Look at each feature independently — understand its distribution before relating it to the target.

### 4a. Age

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(df['Age'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')

# Box plot — shows median, quartiles, and outliers
axes[1].boxplot(df['Age'], vert=False)
axes[1].set_title('Age Box Plot')
axes[1].set_xlabel('Age')

plt.tight_layout()
plt.show()

print(df['Age'].describe())

### 4b. Annual Premium

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['Annual_Premium'], bins=50, color='coral', edgecolor='white')
axes[0].set_title('Annual Premium Distribution (with outliers)')
axes[0].set_xlabel('Annual Premium')

# Remove top 1% to see the core distribution clearly
p99 = df['Annual_Premium'].quantile(0.99)
axes[1].hist(df[df['Annual_Premium'] <= p99]['Annual_Premium'], bins=50, color='coral', edgecolor='white')
axes[1].set_title('Annual Premium (99th percentile cap)')
axes[1].set_xlabel('Annual Premium')

plt.tight_layout()
plt.show()

outlier_count = (df['Annual_Premium'] > 200000).sum()
print(f'Rows with Annual_Premium > 200,000: {outlier_count}  ({outlier_count/len(df)*100:.2f}% of data)')

### 4c. Vintage (days with company)

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df['Vintage'], bins=30, color='mediumpurple', edgecolor='white')
plt.title('Vintage Distribution (Days with Company)')
plt.xlabel('Vintage (days)')
plt.ylabel('Count')
plt.show()

# Almost uniform — customers are spread evenly across tenure
print(df['Vintage'].describe())

### 4d. Categorical Features

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

cat_binary = {
    'Gender': df['Gender'].value_counts(),
    'Driving_License': df['Driving_License'].map({1: 'Has License', 0: 'No License'}).value_counts(),
    'Previously_Insured': df['Previously_Insured'].map({1: 'Already Insured', 0: 'Not Insured'}).value_counts(),
    'Vehicle_Age': df['Vehicle_Age'].value_counts(),
    'Vehicle_Damage': df['Vehicle_Damage'].value_counts(),
}

for ax, (col, counts) in zip(axes.flatten(), cat_binary.items()):
    ax.bar(counts.index, counts.values, color='teal', edgecolor='white')
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=20)
    for i, v in enumerate(counts.values):
        ax.text(i, v + 500, f'{v:,}', ha='center', fontsize=9)

axes[1][2].axis('off')  # hide unused subplot
plt.suptitle('Categorical Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 5 — Feature vs Target Analysis (Bivariate)

Now relate each feature to the target `Response`.  
Goal: **find which features discriminate between interested and not interested customers.**

This tells you which features will be most useful in the model.

### 5a. Age vs Response

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of age for each response class
df[df['Response'] == 0]['Age'].hist(bins=30, ax=axes[0], alpha=0.7, label='Not Interested', color='#e74c3c')
df[df['Response'] == 1]['Age'].hist(bins=30, ax=axes[0], alpha=0.7, label='Interested', color='#2ecc71')
axes[0].set_title('Age Distribution by Response')
axes[0].set_xlabel('Age')
axes[0].legend()

# Average age per response class
avg_age = df.groupby('Response')['Age'].mean()
axes[1].bar(['Not Interested (0)', 'Interested (1)'], avg_age.values, color=['#e74c3c', '#2ecc71'])
axes[1].set_title('Average Age by Response')
axes[1].set_ylabel('Mean Age')
for i, v in enumerate(avg_age.values):
    axes[1].text(i, v - 2, f'{v:.1f}', ha='center', color='white', fontweight='bold')

plt.tight_layout()
plt.show()

print('Mean age — Not Interested:', df[df['Response']==0]['Age'].mean().round(1))
print('Mean age — Interested:    ', df[df['Response']==1]['Age'].mean().round(1))

### 5b. Previously Insured vs Response  
This is the **most important business feature** — someone already insured elsewhere will almost never buy from you.

In [ ]:
cross_tab = pd.crosstab(df['Previously_Insured'], df['Response'], margins=True)
cross_tab.index = ['Not Insured', 'Already Insured', 'Total']
cross_tab.columns = ['Not Interested', 'Interested', 'Total']
print('=== Previously Insured vs Response ===')
print(cross_tab)

# Response rate within each group
rate = df.groupby('Previously_Insured')['Response'].mean() * 100
print('\nResponse rate:')
print(f'  Not Insured (0):      {rate[0]:.1f}% interested')
print(f'  Already Insured (1):  {rate[1]:.1f}% interested')

# Plot
plt.figure(figsize=(6, 4))
rate.plot(kind='bar', color=['#2ecc71', '#e74c3c'])
plt.title('Response Rate by Previously Insured Status')
plt.xlabel('Previously Insured')
plt.ylabel('% Interested in Vehicle Insurance')
plt.xticks([0, 1], ['Not Insured', 'Already Insured'], rotation=0)
plt.show()

### 5c. Vehicle Damage vs Response  
Has the customer's vehicle been damaged before? People who've experienced damage understand why insurance matters.

In [ ]:
rate = df.groupby('Vehicle_Damage')['Response'].mean() * 100
print('Response rate by Vehicle Damage:')
print(rate.round(1))

plt.figure(figsize=(6, 4))
rate.plot(kind='bar', color=['#e74c3c', '#2ecc71'])
plt.title('Response Rate by Vehicle Damage History')
plt.xlabel('Vehicle Damage')
plt.ylabel('% Interested')
plt.xticks(rotation=0)
plt.show()

### 5d. Vehicle Age vs Response

In [ ]:
rate = df.groupby('Vehicle_Age')['Response'].mean() * 100
counts = df.groupby('Vehicle_Age')['Response'].count()

print('Response rate by Vehicle Age:')
summary = pd.DataFrame({'total_customers': counts, 'response_rate_%': rate.round(1)})
print(summary)

# Order properly
order = ['< 1 Year', '1-2 Year', '> 2 Years']
rate_ordered = rate.reindex(order)

plt.figure(figsize=(7, 4))
rate_ordered.plot(kind='bar', color='steelblue')
plt.title('Response Rate by Vehicle Age')
plt.xlabel('Vehicle Age')
plt.ylabel('% Interested')
plt.xticks(rotation=0)
plt.show()

### 5e. Gender vs Response

In [ ]:
rate = df.groupby('Gender')['Response'].mean() * 100
print('Response rate by Gender:', rate.round(2).to_dict())

plt.figure(figsize=(5, 4))
rate.plot(kind='bar', color=['#e91e63', '#1565c0'])
plt.title('Response Rate by Gender')
plt.ylabel('% Interested')
plt.xticks(rotation=0)
plt.show()
print('Observation: Gender has minimal impact on response rate.')

### 5f. Annual Premium vs Response

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df[df['Response']==0]['Annual_Premium'].clip(upper=100000).hist(bins=40, ax=ax, alpha=0.6, label='Not Interested', color='#e74c3c')
df[df['Response']==1]['Annual_Premium'].clip(upper=100000).hist(bins=40, ax=ax, alpha=0.6, label='Interested', color='#2ecc71')
ax.set_title('Annual Premium Distribution by Response (capped at 100K)')
ax.set_xlabel('Annual Premium')
ax.legend()
plt.show()

print('Mean Annual Premium — Not Interested:', df[df['Response']==0]['Annual_Premium'].mean().round(0))
print('Mean Annual Premium — Interested:    ', df[df['Response']==1]['Annual_Premium'].mean().round(0))

---
## Step 6 — Correlation Analysis

After encoding, check which features **numerically correlate** with the target.  
A heatmap shows correlations between all pairs of features.

In [ ]:
# Encode for correlation — we need numbers
df_corr = df.copy()
df_corr['Gender'] = df_corr['Gender'].map({'Male': 1, 'Female': 0})
df_corr['Vehicle_Damage'] = df_corr['Vehicle_Damage'].map({'Yes': 1, 'No': 0})
df_corr['Vehicle_Age'] = df_corr['Vehicle_Age'].map({'< 1 Year': 0, '1-2 Year': 1, '> 2 Years': 2})
df_corr = df_corr.drop('id', axis=1)

# Correlation with target
corr_with_target = df_corr.corr()['Response'].drop('Response').sort_values(ascending=False)
print('=== Correlation with Target (Response) ===')
print(corr_with_target.round(3))

plt.figure(figsize=(7, 5))
corr_with_target.plot(kind='barh', color=['#2ecc71' if v > 0 else '#e74c3c' for v in corr_with_target])
plt.title('Feature Correlation with Target (Response)')
plt.xlabel('Pearson Correlation')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Full correlation heatmap (feature-to-feature)
plt.figure(figsize=(11, 8))
sns.heatmap(df_corr.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

---
## Step 7 — Data Preprocessing

Machine learning models only understand numbers.  
We need to:
1. Encode categorical columns → numbers
2. Scale numeric columns → same range
3. Drop the ID column (not a feature)

**Important rule:** Fit scalers only on training data. Apply (transform) on both train and test.  
Fitting on the full dataset = **data leakage** (test data influences preprocessing).

In [ ]:
df_model = df.copy()

# Drop ID — not a feature
df_model = df_model.drop('id', axis=1)

# 1. Encode Gender
df_model['Gender'] = df_model['Gender'].map({'Male': 1, 'Female': 0})

# 2. Encode Vehicle_Damage
df_model['Vehicle_Damage'] = df_model['Vehicle_Damage'].map({'Yes': 1, 'No': 0})

# 3. One-hot encode Vehicle_Age (3 categories -> 2 columns, drop_first avoids redundancy)
df_model = pd.get_dummies(df_model, columns=['Vehicle_Age'], drop_first=True)

# Rename for clarity
df_model.rename(columns={
    'Vehicle_Age_1-2 Year': 'Vehicle_Age_1_2_Year',
    'Vehicle_Age_> 2 Years': 'Vehicle_Age_gt_2_Years'
}, inplace=True)

print('Columns after encoding:', df_model.columns.tolist())
df_model.head(3)

---
## Step 8 — Train-Test Split

Split before scaling — this prevents the test set from influencing the scaler (data leakage).

In [ ]:
X = df_model.drop('Response', axis=1)
y = df_model['Response']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y   # ensures same class ratio in both splits
)

print(f'Train size: {X_train.shape[0]:,}  ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Test size:  {X_test.shape[0]:,}  ({X_test.shape[0]/len(X)*100:.1f}%)')
print(f'\nClass ratio in train: {y_train.mean()*100:.1f}% positive')
print(f'Class ratio in test:  {y_test.mean()*100:.1f}% positive')

In [ ]:
# Now scale — fit ONLY on train, transform both train and test
num_cols = ['Age', 'Vintage']
premium_col = ['Annual_Premium']

ss = StandardScaler()
X_train[num_cols] = ss.fit_transform(X_train[num_cols])   # fit + transform on train
X_test[num_cols]  = ss.transform(X_test[num_cols])         # transform only on test

mm = MinMaxScaler()
X_train[premium_col] = mm.fit_transform(X_train[premium_col])
X_test[premium_col]  = mm.transform(X_test[premium_col])

print('Scaling done. Train sample:')
X_train.head(2)

---
## Step 9 — Model Training & Evaluation

We start with a **Random Forest** baseline.  
Important: with imbalanced data, look at **ROC-AUC and F1**, not accuracy.

In [ ]:
# Train with class_weight='balanced' — tells the model to penalize
# missing class-1 predictions more (handles imbalance without resampling)
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',   # KEY: handles the 88/12 imbalance
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
print('Model trained.')

In [ ]:
y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]  # probability of class 1

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Not Interested', 'Interested']))

print(f'ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}')
print('(ROC-AUC > 0.7 is acceptable, > 0.8 is good for imbalanced data)')

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Interested', 'Interested'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (correct Not Interested): {tn:,}')
print(f'False Positives (wrongly predicted Yes):  {fp:,}')
print(f'False Negatives (missed actual Yes):      {fn:,}  <-- business cost: missed leads')
print(f'True Positives  (correct Interested):     {tp:,}')

### Feature Importance  
Which features did the model find most useful?

In [ ]:
feat_imp = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
feat_imp.plot(kind='bar', color='steelblue')
plt.title('Feature Importance (Random Forest)')
plt.ylabel('Importance Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('Top 5 most important features:')
print(feat_imp.head())

---
## Step 10 — Summary & What to Fix Before Production

### What we did
| Step | What | Why |
|---|---|---|
| Data Quality | Null check, duplicate check, value ranges | Catch bad data early |
| Target Analysis | Class distribution | Revealed 88/12 imbalance |
| Univariate EDA | Each feature independently | Understand distributions, spot outliers |
| Bivariate EDA | Each feature vs target | Find which features drive the prediction |
| Correlation | Heatmap | Check multicollinearity |
| Encoding | Label + One-Hot | Convert categories to numbers |
| Scaling | StandardScaler + MinMaxScaler | Normalize numeric ranges |
| Train-Test Split | 75/25, stratified | Evaluate on unseen data |
| Modeling | Random Forest + class_weight | Baseline model |
| Evaluation | F1, ROC-AUC, Confusion Matrix | Correct metrics for imbalanced data |

### Key Findings
- **`Previously_Insured`** is the strongest signal — customers already insured elsewhere almost never respond
- **`Vehicle_Damage`** is the second strongest — prior damage makes customers value insurance
- **`Vehicle_Age`** matters — older vehicles have higher response rates
- **`Driving_License`** is nearly useless — 99.8% of customers already have one
- **`Vintage`** has almost no correlation with response

### What to improve
1. Try **SMOTE** (Synthetic Minority Oversampling) for better class-1 recall
2. Try **XGBoost / LightGBM** — typically outperform Random Forest on tabular data
3. Cap outliers in `Annual_Premium` before scaling
4. Use **Precision-Recall curve** to pick the right threshold for your business use case
5. Wrap everything into a `sklearn.Pipeline` for clean, leakage-free deployment